# Johnson Noise Propagation Through a Readout Chain

This notebook demonstrates Johnson noise propagation through a cascaded readout chain
typical of quantum device measurements, reproducing the analysis from Figure 7.4 of Wen's thesis.

## Background

**Johnson noise** (thermal noise) is the electronic noise generated by thermal agitation of
charge carriers in a conductor at equilibrium. The noise power spectral density is:

$$P = k_B T \Delta f$$

where $k_B$ is Boltzmann's constant, $T$ is temperature, and $\Delta f$ is bandwidth.

We express all noise quantities as **equivalent noise temperatures**, so a noise power $P$
corresponds to temperature $T = P / (k_B \Delta f)$.

## Friis Formula

For a cascade of components with gains $G_1, G_2, \ldots$ and added noise temperatures
$T_1, T_2, \ldots$ (referred to each component's input), the total system noise temperature
referred to the input is:

$$T_\text{sys} = T_1 + \frac{T_2}{G_1} + \frac{T_3}{G_1 G_2} + \cdots$$

This shows that the **first components dominate** the system noise — hence the importance of
low-noise amplifiers close to the detector in cryogenic setups.

In [1]:
from noise import Component, ReadoutChain, analyze_chain
from noise.plotting import plot_chain_analysis

## Define the readout chain

As a calibration, we try to reproduce Osmond Wen thesis Fig. 7.4.

A typical KID readout chain from 300 K input down to the 50 mK detector stage and back up through amplifiers:

| Component | Gain | Temperature | Notes |
|-----------|------|-------------|-------|
| 20 dB attenuator | -20 dB | 4 K | First stage cooling |
| 10 dB attenuator | -10 dB | 1 K | Second stage |
| 10 dB attenuator | -10 dB | 50 mK | Mixing chamber |
| KID | 0 dB | 50 mK | Detector (passive) |
| HEMT | +40 dB | 4 K | Low-noise cryogenic amplifier, T_n = 5 K |
| Warm amplifier | +25 dB | 300 K | Room-temperature amplifier, T_n = 50 K |

In [2]:
chain = (
    ReadoutChain(signal_temp_K=300.0)
    .add(Component("20dB atten",        gain_dB=-20.0, temperature_K=4.0))
    .add(Component("10dB atten (1K)",   gain_dB=-10.0, temperature_K=1.0))
    .add(Component("10dB atten (50mK)", gain_dB=-10.0, temperature_K=50e-3))
    .add(Component("KID",               gain_dB=0.0,   temperature_K=50e-3))
    .add(Component("HEMT",              gain_dB=40.0,  temperature_K=4.0, noise_temp_K=5.0))
    .add(Component("Warm amp",          gain_dB=25.0,  temperature_K=300.0, noise_temp_K=50.0))
)

## Run the noise analysis

`analyze_chain()` walks the chain left-to-right, tracking the signal and every individual noise contribution at each stage boundary. It also computes the Friis system noise temperature.

In [3]:
analysis = analyze_chain(chain)

print(f"System noise temperature (Friis): {analysis.friis_noise_temp:.1f} K")
print(f"\nBack-referred noise contributions:")
for name, temp in analysis.back_referred.items():
    print(f"  {name:25s} {temp:.4f} K")

System noise temperature (Friis): 51796.0 K

Back-referred noise contributions:
  20dB atten                396.0000 K
  10dB atten (1K)           900.0000 K
  10dB atten (50mK)         450.0000 K
  KID                       0.0000 K
  HEMT                      50000.0000 K
  Warm amp                  50.0000 K


## Stage-by-stage breakdown

In [4]:
stage_labels = ["Input"] + [c.name for c in chain.components]

for stage in analysis.stages:
    label = stage_labels[stage.stage_index]
    print(f"--- {label} (stage {stage.stage_index}) ---")
    print(f"  Signal:      {stage.signal_temp:.6f} K")
    for name, val in stage.noise_contributions.items():
        print(f"  Noise [{name:25s}]: {val:.6f} K")
    print(f"  Total noise: {stage.total_noise:.6f} K")
    print()

--- Input (stage 0) ---
  Signal:      300.000000 K
  Total noise: 0.000000 K

--- 20dB atten (stage 1) ---
  Signal:      3.000000 K
  Noise [20dB atten               ]: 3.960000 K
  Total noise: 3.960000 K

--- 10dB atten (1K) (stage 2) ---
  Signal:      0.300000 K
  Noise [20dB atten               ]: 0.396000 K
  Noise [10dB atten (1K)          ]: 0.900000 K
  Total noise: 1.296000 K

--- 10dB atten (50mK) (stage 3) ---
  Signal:      0.030000 K
  Noise [20dB atten               ]: 0.039600 K
  Noise [10dB atten (1K)          ]: 0.090000 K
  Noise [10dB atten (50mK)        ]: 0.045000 K
  Total noise: 0.174600 K

--- KID (stage 4) ---
  Signal:      0.030000 K
  Noise [20dB atten               ]: 0.039600 K
  Noise [10dB atten (1K)          ]: 0.090000 K
  Noise [10dB atten (50mK)        ]: 0.045000 K
  Noise [KID                      ]: 0.000000 K
  Total noise: 0.174600 K

--- HEMT (stage 5) ---
  Signal:      300.000000 K
  Noise [20dB atten               ]: 396.000000 K
  Noise

## Plot: Signal and noise propagation (Figure 7.4 style)

The plot shows equivalent noise temperatures at each stage boundary on a log scale:
- **Black bars**: signal level
- **Colored bars**: individual noise contributions from each component
- **Gold bars**: total noise power
- **Faded bars**: noise components back-referred to each stage

In [ ]:
%matplotlib inline
fig = plot_chain_analysis(chain, analysis, bandwidth_Hz=1e6)
fig.savefig("johnson_noise_chain.png", dpi=150, bbox_inches="tight")

## Unit conversion: temperature ↔ dBm

Noise power in dBm and equivalent noise temperature are related by:

$$P_\text{dBm} = 10 \log_{10}\!\left(\frac{k_B \, T \, \Delta f}{1\;\text{mW}}\right)$$

The helper functions `temp_to_dBm` and `dBm_to_temp` perform this conversion given a bandwidth.

In [6]:
from noise import temp_to_dBm, dBm_to_temp

bandwidth_Hz = 1e6  # 1 MHz readout bandwidth

# Convert stage-by-stage results to dBm
print(f"Signal and noise levels in dBm (bandwidth = {bandwidth_Hz/1e6:.0f} MHz)\n")
stage_labels = ["Input"] + [c.name for c in chain.components]

for stage in analysis.stages:
    label = stage_labels[stage.stage_index]
    sig_dBm = temp_to_dBm(stage.signal_temp, bandwidth_Hz)
    print(f"--- {label} ---")
    print(f"  Signal: {stage.signal_temp:12.4f} K = {sig_dBm:8.2f} dBm")
    if stage.total_noise > 0:
        noise_dBm = temp_to_dBm(stage.total_noise, bandwidth_Hz)
        print(f"  Noise:  {stage.total_noise:12.4f} K = {noise_dBm:8.2f} dBm")
    print()

Signal and noise levels in dBm (bandwidth = 1 MHz)

--- Input ---
  Signal:     300.0000 K =  -113.83 dBm

--- 20dB atten ---
  Signal:       3.0000 K =  -133.83 dBm
  Noise:        3.9600 K =  -132.62 dBm

--- 10dB atten (1K) ---
  Signal:       0.3000 K =  -143.83 dBm
  Noise:        1.2960 K =  -137.47 dBm

--- 10dB atten (50mK) ---
  Signal:       0.0300 K =  -153.83 dBm
  Noise:        0.1746 K =  -146.18 dBm

--- KID ---
  Signal:       0.0300 K =  -153.83 dBm
  Noise:        0.1746 K =  -146.18 dBm

--- HEMT ---
  Signal:     300.0000 K =  -113.83 dBm
  Noise:    51746.0000 K =   -91.46 dBm

--- Warm amp ---
  Signal:   94868.3298 K =   -88.83 dBm
  Noise:  16379333.3686 K =   -66.46 dBm



In [7]:
# Round-trip demonstration
print("Round-trip: dBm → K → dBm")
for p in [-120, -100, -80, -60]:
    t = dBm_to_temp(p, bandwidth_Hz)
    p_back = temp_to_dBm(t, bandwidth_Hz)
    print(f"  {p:+4d} dBm  →  {t:.4e} K  →  {p_back:+.2f} dBm")

Round-trip: dBm → K → dBm
  -120 dBm  →  7.2430e+01 K  →  -120.00 dBm
  -100 dBm  →  7.2430e+03 K  →  -100.00 dBm
   -80 dBm  →  7.2430e+05 K  →  -80.00 dBm
   -60 dBm  →  7.2430e+07 K  →  -60.00 dBm


## Noise figure ↔ noise temperature

Amplifier noise is sometimes specified as a **noise figure** (NF) in dB rather than a noise temperature. The conversion uses the IEEE standard reference temperature $T_0 = 290$ K:

$$T_n = T_0 \left(10^{NF/10} - 1\right) \qquad \Longleftrightarrow \qquad NF = 10 \log_{10}\!\left(1 + \frac{T_n}{T_0}\right)$$

In [8]:
from noise import noise_figure_to_temp, temp_to_noise_figure

# Example: typical noise figures for common amplifiers
print("Noise figure → noise temperature:")
for nf in [0.5, 1.0, 1.5, 3.0, 6.0]:
    t = noise_figure_to_temp(nf)
    print(f"  NF = {nf:.1f} dB  →  T_n = {t:.1f} K")

print("\nNoise temperature → noise figure:")
for t in [5, 50, 290, 1000]:
    nf = temp_to_noise_figure(t)
    print(f"  T_n = {t:4d} K   →  NF = {nf:.2f} dB")

# Using noise_figure_to_temp to define a component
print(f"\nExample: an LNA with NF = 1.5 dB has T_n = {noise_figure_to_temp(1.5):.1f} K")
print("  Component('LNA', gain_dB=30, temperature_K=300,")
print(f"            noise_temp_K=noise_figure_to_temp(1.5))  # {noise_figure_to_temp(1.5):.1f} K")

Noise figure → noise temperature:
  NF = 0.5 dB  →  T_n = 35.4 K
  NF = 1.0 dB  →  T_n = 75.1 K
  NF = 1.5 dB  →  T_n = 119.6 K
  NF = 3.0 dB  →  T_n = 288.6 K
  NF = 6.0 dB  →  T_n = 864.5 K

Noise temperature → noise figure:
  T_n =    5 K   →  NF = 0.07 dB
  T_n =   50 K   →  NF = 0.69 dB
  T_n =  290 K   →  NF = 3.01 dB
  T_n = 1000 K   →  NF = 6.48 dB

Example: an LNA with NF = 1.5 dB has T_n = 119.6 K
  Component('LNA', gain_dB=30, temperature_K=300,
            noise_temp_K=noise_figure_to_temp(1.5))  # 119.6 K
